In [1154]:
from game import Game
from collections import deque
import time
import os
import re
import heapq

## Load maps

In [1155]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [1156]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [1157]:
game.get_box_locations()
game.get_portal_locations()

[[(0, 1), (3, 4)]]

In [1158]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [1159]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [1160]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [1161]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [1162]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

### State definition

In [1163]:
class State:
    def __init__(self, box_locations, player_position):
        self.box_locations = tuple(box_locations)
        self.player_position = player_position
    def __eq__(self, other):
        return (self.box_locations == other.box_locations) and (self.player_position == other.player_position)
    def __lt__(self, other): 
        return self.player_position < other.player_position
    def __hash__(self):
        return hash((self.box_locations, self.player_position))


### BFS

In [1164]:
# TODO: Must return moves (if there is no solution return None), number of visited states
def is_locked_unsolved_box(game): 
    box_locations = game.get_box_locations()
    goal_locations = game.get_goal_locations()
    for i in range(len(box_locations)): 
        if (box_locations[i] == goal_locations[i]): 
            continue
        x = box_locations[i][0]
        y = box_locations[i][1] 
        if ((game.is_wall((x + 1, y)) and game.is_wall((x, y - 1))) or (game.is_wall((x + 1, y)) and game.is_wall((x, y + 1))) or (game.is_wall((x - 1, y)) and game.is_wall((x, y + 1))) or (game.is_wall((x - 1, y)) and game.is_wall((x, y - 1)))): 
            return True
    return False

def build_solution(id, parent) :
    answer = [] 
    while True : 
        if id == 0 : 
            break
        answer.append(parent[id][1])
        id = parent[id][0]
    answer.reverse()
    return ''.join(answer)

def solver_bfs(game_map):
    game = Game(game_map)
    initial_state = State(game.get_box_locations(), game.get_player_position())
    visited = set()
    queue = deque([(initial_state, 0)])
    parent = [(-1, None)]
    visited.add(initial_state)
    new_id = 0
    while len(queue) > 0: 
        current_state, id = queue.popleft()
        for direction in ['U', 'D', 'R', 'L']:
            game.set_box_positions(current_state.box_locations)
            game.set_player_position(current_state.player_position)
            result = game.apply_move(direction)
            if result:
                next_state = State(game.get_box_locations(), game.get_player_position())
                if next_state not in visited: 
                    if (is_locked_unsolved_box(game)): 
                        visited.add(next_state)
                        continue 
                    visited.add(next_state)
                    new_id += 1
                    parent.append((id, direction))
                    queue.append((next_state, new_id))
                    if game.is_game_won(): 
                        return build_solution(new_id, parent), len(visited)
        
    return None, len(visited)

### DFS

In [1165]:
# TODO: Must return moves, number of visited states
directions = ['U', 'D', 'R', 'L']

def build_solution_dfs(stack): 
    answer = [] 
    for state, direction_id in stack : 
        answer.append(directions[direction_id - 1])
    return ''.join(answer)

def solver_dfs(game_map):
    game = Game(game_map)
    initial_state = State(game.get_box_locations(), game.get_player_position())
    stack = [(initial_state, 0)]
    visited = set() 
    visited.add(initial_state)
    number_of_visisted_states = 0
    while len(stack) > 0: 
        current_state, direction_id = stack.pop() 
        number_of_visisted_states += (direction_id == 0) 
        if (direction_id > 3) : 
            visited.remove(current_state)
            continue
        stack.append((current_state, direction_id + 1))
        game.set_box_positions(current_state.box_locations)
        game.set_player_position(current_state.player_position)
        result = game.apply_move(directions[direction_id])
        if result: 
            next_state = State(game.get_box_locations(), game.get_player_position())
            if next_state not in visited: 
                if (is_locked_unsolved_box(game)): 
                        visited.add(next_state)
                        continue 
                if game.is_game_won(): 
                    return build_solution_dfs(stack), number_of_visisted_states
                stack.append((next_state, 0))
                visited.add(next_state)

    return None, number_of_visisted_states

### IDS

In [1166]:
# TODO: Must return moves, number of visited states

directions = ['U', 'D', 'R', 'L']

def build_solution_ids(stack): 
    answer = [] 
    for state, direction_id in stack : 
        answer.append(directions[direction_id - 1])
    return ''.join(answer)

def solver_ids(game_map):
    max_depth = 1
    previous_number_of_visited_states = 0 
    while True: 
        game = Game(game_map)
        initial_state = State(game.get_box_locations(), game.get_player_position())
        stack = [(initial_state, 0)]
        visited = set() 
        visited.add(initial_state)
        number_of_visisted_states = 0
        while len(stack) > 0: 
            current_state, direction_id = stack.pop() 
            number_of_visisted_states += (direction_id == 0) 
            if (direction_id > 3) : 
                visited.remove(current_state)
                continue
            if (max_depth == len(stack)) : 
                visited.remove(current_state)
                continue
            stack.append((current_state, direction_id + 1))
            game.set_box_positions(current_state.box_locations)
            game.set_player_position(current_state.player_position)
            result = game.apply_move(directions[direction_id])
            if result: 
                next_state = State(game.get_box_locations(), game.get_player_position())
                if next_state not in visited: 
                    if (is_locked_unsolved_box(game)): 
                        visited.add(next_state)
                        continue 
                    if game.is_game_won(): 
                        return build_solution_dfs(stack), number_of_visisted_states
                    stack.append((next_state, 0))
                    visited.add(next_state)
        if (number_of_visisted_states == previous_number_of_visited_states): 
            break
        previous_number_of_visited_states = number_of_visisted_states
        max_depth += 1 
    return None, number_of_visisted_states

### A*

In [1167]:
# TODO

def get_distance(start, end): 
    return abs(start[0] - end[0]) + abs(start[1] - end[1]) 

def heuristic(game):
    box_locations = game.get_box_locations()
    goal_locations = game.get_goal_locations()
    manhatan_distance = 0
    portal_locations = game.get_portal_locations()
    for i in range(len(box_locations)):
        box = box_locations[i]
        goal = goal_locations[i]
        min_distance = get_distance(box, goal)
        for portal in portal_locations: 
            min_distance = min(min_distance, get_distance(box, portal[0]) + get_distance(portal[1], goal) - 1)
            min_distance = min(min_distance, get_distance(box, portal[1]) + get_distance(portal[0], goal) - 1)
        if len(portal_locations) == 2: 
            portal1 = portal_locations[0]
            portal2 = portal_locations[1]
            min_distance = min(min_distance, get_distance(box, portal1[0]) + get_distance(portal1[1], portal2[0]) + get_distance(portal2[1], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal1[1]) + get_distance(portal1[0], portal2[0]) + get_distance(portal2[1], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal1[0]) + get_distance(portal1[1], portal2[1]) + get_distance(portal2[0], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal1[1]) + get_distance(portal1[0], portal2[1]) + get_distance(portal2[0], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal2[0]) + get_distance(portal2[1], portal1[0]) + get_distance(portal1[1], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal2[1]) + get_distance(portal2[0], portal1[0]) + get_distance(portal1[1], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal2[0]) + get_distance(portal2[1], portal1[1]) + get_distance(portal1[0], goal) - 2)
            min_distance = min(min_distance, get_distance(box, portal2[1]) + get_distance(portal2[0], portal1[1]) + get_distance(portal1[0], goal) - 2)
        manhatan_distance += min_distance
    return manhatan_distance



def build_solution_astar(parent, state): 
    answer = [] 
    while True: 
        if (parent[state][0] is None): 
            break 
        answer.append(parent[state][1])
        state = parent[state][0] 
    answer.reverse()
    return ''.join(answer)

# TODO: Must return moves, number of visited states
directions = ['U', 'D', 'R', 'L']
def solver_astar(game_map, heuristic_func=heuristic, weight=1):
    game = Game(game_map)
    heap = [] 
    initial_state = State(game.get_box_locations(), game.get_player_position())
    heapq.heappush(heap, (heuristic_func(game) * weight, 0, initial_state, None, None))
    visited = set() 
    parent = {}
    parent[initial_state] = (None, None)
    num_visited_states = 0 
    while heap: 
        current_f_value, current_g_value, current_state, parent_state, parent_char = heapq.heappop(heap)
        if (current_state in visited): 
            continue
        parent[current_state] = (parent_state, parent_char)
        visited.add(current_state)
        game.set_box_positions(current_state.box_locations)
        game.set_player_position(current_state.player_position)
        if game.is_game_won(): 
            return build_solution_astar(parent, current_state), len(visited)
        for direction in ['U', 'D', 'R', 'L']:
            game.set_box_positions(current_state.box_locations)
            game.set_player_position(current_state.player_position)
            result = game.apply_move(direction)
            if result: 
                next_state = State(game.get_box_locations(), game.get_player_position())
                if next_state not in visited:
                    if (is_locked_unsolved_box(game)): 
                        visited.add(next_state)
                        continue 
                    next_g_value = current_g_value + 1 
                    next_f_value = next_g_value + heuristic_func(game) * weight
                    heapq.heappush(heap, (next_f_value, next_g_value, next_state, current_state, direction))
    return None, len(visited)

## Solve

In [1168]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar,
    "Weighted A*": solver_astar
}

In [1169]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves = 0 
    numof_visited_states = 0
    if (method == "Weighted A*"): 
        moves, numof_visited_states = SOLVERS[method](game_map, heuristic, 15)
    else : 
        moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

# سوال اول
در این بازی میتوانیم استیت ها را موقعیت جعبه ها و موفعیت پلیر در نظر گرفت. بدیهی ترین تعریف این است که یک ورژن از نقشه را در نظر بگیریم که که هم مموری زیادی نیاز دارد هم هنگام استفاده از آن سربار زمانی زیادی میدهیم. اگر دقت کنیم مثلا لوکیشن هدف ها یا پرتال ها یا دیوار ها یا... در همه استیت ها یکسان است و تنها میتوانیم قسمت های متغیر را به عنوان پارامتری از استیت ها در نظر بگیریم. 

# سوال دوم
هر اکشن را میتوانیم حرکت پلیر در یکی از 4 جهت اصلی به میزان یک واحد در نظر گرفت که ما را از استیتی به استیت  دیگر منتقل میکند(ممکن است در همان استیت هم بمانیم) 

# سوال سوم
با توجه به تعریفی که برای استیت و اکشن ارائه دادیم، استیت اینیشیال را همان استیت اولیه نقشته ها در نظر میگیریم که هنوز هیچ اکشنی انجام ندادیم. و استیت هدف را هم استیتی در نظر میگیریم که تمام جعبه ها در جایگاه هدف قرار گرفته باشند. به عبارت دیگر یک استیت هدف است اگر و فقط اگر تابع IS_GAME_WON() TRUE باشد.

اکشن ها را هم با کاراکتر های L, R ,D ,U مدل کردیم که نمایانگر یه واحد حرکت در 4 جهت اصلی است. 

# سوال چهارم
بهینه سازی هایی که من انجام دادم، یکی بخش تعریف استیت هاست که تنها متغیر های بازی در هر حرکت را درنظر گرفتم یعنی مکان جعبه ها و بازیکن. بهینه سازی دیگری که داشتم استفاده از ست به عنوان داده ساختاری برای نگه داشتن استیت های دیده شده بود. با اینکار اولا در استیت تکراری نمیرویم و دومن چون که ست استیت ها را هش میکند، اردر زمانی خیلی کمتری استفاده از ان دارد و میتوانیم در زمان خوبی وجود یک استیت را بررسی کنیم. بهینه سازی بعدی که انجام دادم این بود که در یک سری از حالت ها از گسترش اضافی درخت یعنی حالاتی که هیچ وقت به هدف نمیرسند جلوگیری کردم. این حالات حالاتی هستند که در دو جهت مجاور از 4 جهت حرکت دیوار قرار دارد و دیگر نمیتوان جعبه را هل داد و قفل میشود. اگر جعبه قفل شود و در جایگاه هدفش نباشد دیگر ادامه این حالت را گسترش نمیدهیم. اگر اینکار را نکنیم و تمام حالات را ببینیم واضحا هم زمان خیلی بیشتری مصرف میکنیم هم حافظه بیشتری مصرف میکنیم. 

# سوال پنجم
## بی اف اس
الگوریتم بی اف اس به ازای اینیشیال استیت کوتاه ترین مسیر را نسبت به بقیه استیت ها پیدا میکند برای همین بهینه است اما مموری خیلی زیادی دارد چون باید تمام استیت ها را در یک ست ذخیره کنیم تا وارد استیت تکراری نشویم. مزیت آن این است که اگر تعداد حالات کم باشد یا عمق استیت هدف بدانیم کم است، این الگوریتم به خوبی کار میکند. 

## دی اف اس
این الگوریتم جست و جو در عمق انجام میدهد و صرفا مسیری که طی کرده را در حافظه خود نگه میدارد، از این رو حافظه خوبی دارد اما به دلیل اینکه تعداد مسیر های ممکن خیلی زیاد است خیلی طول میکشد تا جواب را پیدا کند. اکثر اوقات هم جوابی پیدا نمیکند چون در نقشه های ساده فقط قادر به پیدا کردن جواب هست و لزومن جواب بهینه نیست. از مزیت دیگر این الگورتیم به پیاده سازی ساده آن نیز میتوان اشاره کرد. 

## آی دی اس
این الگوریتم تلفیقی از بی اف اس و دی اف اس است که انگار الگوریتم دی اف اس را مرتبن مانند بی اف اس اجرا میکند. یعنی ابتدا تمام مسیر های به طول یک سپس تمام مسیر های به طول دو بعد سه بعد چهار ... را انجام میدهد و تمام مسیر های دی اف اسی با حداکثر آن عمق را میبیند تا استیت هدف را پیدا کند. مزیت این الگوریتم این است که اولا جواب بهینه را پیدا میکند و ثانیا حافظه ای مشابه با حافظه دی اف اس دارد چون در عمل الگوریتم دی اف اس را مرتبا اجرا میکند و مزیت های دی اف اس و بی اف اس را شامل میشود. اما یکی از معایب آن این است که هر بار که حداکثر ارتفاع را یکی زیاد میکند تمام مسیر هایی که در لول قبلی دیده شده دوباره دیده میشوند از این رو باز یکم کند تر از بی اف اس است ولی سریعتر از دی اف اس به جواب میرسد. این الگوریتم در هدف هایی که در عمق بزرگی هستند در زمان معقول نمیتواند جواب بدهد و بدتر از بی اف اس عمل میکند چون که مسیر های تکراری زیادی میبیند 

## آ استار
این الگوریتم یک هیوریستیک دارد که میزان نزدیک بودن به هدف را تخمین میزند و این تخمین با هزینه ای که تا الان صرف شده جمع میشود و به صورت حریصانه شبیه به دایسترا بر حسب جمع تابع جی و اچ تابع اف به دست می آید و از کوچکترین اف استیت ها دیده میشوند و استیت های همسایه هر استیتی که کاملا دیده میشود آپدیت میشود چون ممکن است مسیر بهینه تری پیدا شود. این الگوریتم یک سرچ آگاهانه است و در مپ های بزرگ عملکرد خیلی بهتری نسبت به باقی الگوریتم ها دارد چرا که استیت هدف را خیلی سریع تر پیدا میکند و نحوه تعریف تابع هیوریستیک خیلی در عملکرد آن تاثیر دارد. حالتی هم که تابع اچ را 0 در نظر بگیریم الگورتیمی معادل یو سی اس خواهیم داشت. یک نسخه وزن دار این الگوریتم هم وجود دارد که یک وزن در تابع هیوریستیک ضرب میشود تا بیشتر در راستای هدف متمایل شود. این الگوریتم لزوما جواب بهینه را ممکن است پیدا نکند ولی سریعتر از باقی الگوریتم ها میتواند یک جواب پیدا کند.
الگوریتم آ استار معمولی جواب بهینه را پیدا میکند اما الگوریتم وزن دار آن لزوما جواب بهینه را پیدا نمیکند چرا که شرط های admissible بودن و consistent بودن دیگر برقرار نیست. 

# سوال ششم
نتایج را در سوال 9 به صورت جدول می آورم. همانطور که میبینیم الگوریتم ها مشابه با همان جدول نمونه ای که برای محدودیت زمانی اجرا در صورت پروژه هست شده(کمی بهتر) و مپ 9 رو نتوانسته اجرا کنه. 
اما دی اف اس برخلاف بی اف اس تنها مپ های 1 تا 4 را توانسته اجرا کنه و جواب پیدا کنه. چون مپ های خیلی محدودی هستن و جواب به راحتی پیدا میشود. اما دی اف اس در مپ های بزرگتر که جواب در عمق کوچکتری قرار داشته هم جواب پیدا کرده. 
الگوریتم آی دی اس که تلفیقی از بی اف اس و دی اف اس است تنها برای مپ شماره 8 علاوه بر مپ های 1 تا 5 توانسته جواب بهینه پیدا بکنه. دلیل آن هم اینه که جواب در عمق کمتری نسبت به بقیه مپ ها قرار داره و در عر ض چند ثانیه جواب یافت میشود.
الگوریتم آ استار اما خیلی قدرتمند ظاهر شده و به جز مپ 9 برای تمام مپ ها جواب در زمان معقول پیدا میکند که با تعریف یک هیوریستیک درست میتواند بهینه هم باشد.  پس میبینیم که این الگوریتم عملکرد خوبی در مپ های بزرگ دارد. 
آ استار وزندار حتی برای مپ 9 هم جواب میدهد و خیلی سریع هم به جواب میرسد اما جواب بهینه نیست. 

# سوال هفتم
در این بخش هیوریستیکی که تعریف کردم به این شرح است. به ازای هر جعبه فاصله منهتنی آن را تا جایگاه هدف محاسبه کردم. اما به جای اینکه تنها مسیر مستقیم را در نظر بگیرم تمام حالات استفاده از پرتال ها را هم در نظر گرفتم. یعنی استفاده از یک پرتال یا هر دوپرتال. مینیمم فاصله منهتنی را به ازای هر جعبه به دست اوردم و مجموع همه این ها را به عنوان خروجی هیوریستیک دادم. 
به نظر من این هیوریستیک هم admissible است و هم consistent. 
admissible است زیرا که هیوریستیکی که محاسبه میشود واقعا یک کران پایین برای تعداد حرکات حرکات است چرا که حداقل حرکت جعبه ها در نظر گرفته شده اما جا به جایی پلیر در نظر گرفته نشده... 
برای همین تابع هیوریستیک کمتر از هزینه واقعی است. 
برای consistent بودن هم باید ببینیم که به ازای هر ترنزیشنی که وجود داره آیا اختلاف مقادیر هیوریستیک استیت مبدا و مقصد کوچکتر از هزینه واقعی مبدا تا مقصد هست یا نه؟ بنظرم جواب بله هست چون که این هیوریستیک انگار معادل فاصله واقعی از یک مبدا تا یک مقصد است. از طرفی تابع هیوریستیک ما چون که واقعا یک بخش حقیقی از هزینه واقعی است(صرفا جا به جایی پلیر را ندارد) جابه جایی ها واضحا کوچکتر مساوی هزینه اصلی است. پس درست است. پس این هیوریستیک ما جواب بهینه را میدهد. همچنین از این جهت که consistent هست میفهمیم که admissible هم هست. 

# سوال هشتم
من ابتدا هیورسیتیک عادی که فاصبه منهتنی بدون در نظر گرفتن پرتال بود را اجرا کردم و مدت زمان زیادی طول میکشید تا جواب بدهد. مپ 9 را مثلا در حالتی که وزن دار بود در حدود50 ثانیه جواب میداد. اما در حالتی که پرتال ها را هم در محاسبه فاصله منهتنی در نظر گرفتیم زمان آن پیشرفت خیلی خوبی کرد و در کمتر از 1 ثانیه با ضریب 15 که با سعی و خطا به دست آوردم جواب برای مپ 9 پیدا کردم. 

# سوال نهم

## مپ 1

In [1170]:
solve(1, "BFS")
print("BFS average time: ",0) # 0 ,0 ,0
solve(1, "DFS")
print("DFS average time: ", 0) # 0 ,0 ,0
solve(1, "IDS")
print("IDS average time: ", 0) # 0 ,0 ,0
solve(1, "A*")
print("A* average time: ", 0) # 0 ,0 ,0
solve(1, "Weighted A*")
print("Weighted A* average time: ", 0) # 0 ,0 ,0


BFS took 0.0 seconds on map 1 and visited 44 states.
7 moves were used: UDDURLL
BFS average time:  0
DFS took 0.0 seconds on map 1 and visited 10 states.
7 moves were used: UDDURLL
DFS average time:  0
IDS took 0.0 seconds on map 1 and visited 10 states.
7 moves were used: UDDURLL
IDS average time:  0
A* took 0.0 seconds on map 1 and visited 44 states.
7 moves were used: DURLLRU
A* average time:  0
Weighted A* took 0.0 seconds on map 1 and visited 14 states.
7 moves were used: UDLRRLD
Weighted A* average time:  0


In [1171]:
solve(2, "BFS")
print("BFS average time: ",0) # 0 ,0 ,0
solve(2, "DFS")
print("DFS average time: ", 0) # 0 ,0 ,0
solve(2, "IDS")
print("IDS average time: ", 0) # 0 ,0 ,0
solve(2, "A*")
print("A* average time: ", 0) # 0 ,0 ,0
solve(2, "Weighted A*")
print("Weighted A* average time: ", 0) # 0 ,0 ,0


BFS took 0.0 seconds on map 2 and visited 22 states.
6 moves were used: LUDDUL
BFS average time:  0
DFS took 0.0 seconds on map 2 and visited 8 states.
6 moves were used: LUDDUL
DFS average time:  0
IDS took 0.0 seconds on map 2 and visited 8 states.
6 moves were used: LUDDUL
IDS average time:  0
A* took 0.0 seconds on map 2 and visited 21 states.
6 moves were used: LDULRU
A* average time:  0
Weighted A* took 0.0 seconds on map 2 and visited 10 states.
6 moves were used: LUDLRD
Weighted A* average time:  0


In [1172]:
solve(3, "BFS")
print("BFS average time: ",0) # 0 ,0 ,0
solve(3, "DFS")
print("DFS average time: ", 0) # 0 ,0 ,0
solve(3, "IDS")
print("IDS average time: ", 0.01) # 0.01 ,0.01 ,0.01
solve(3, "A*")
print("A* average time: ", 0) # 0 ,0 ,0
solve(3, "Weighted A*")
print("Weighted A* average time: ", 0) # 0 ,0 ,0

BFS took 0.0 seconds on map 3 and visited 98 states.
13 moves were used: ULDDUUUURDDDD
BFS average time:  0
DFS took 0.0 seconds on map 3 and visited 57 states.
33 moves were used: ULUURDULDDDUUURDDUULDDDDUUUURDDDD
DFS average time:  0
IDS took 0.01 seconds on map 3 and visited 177 states.
13 moves were used: ULDDUUUURDDDD
IDS average time:  0.01
A* took 0.0 seconds on map 3 and visited 67 states.
13 moves were used: ULDDUUUURDDDD
A* average time:  0
Weighted A* took 0.0 seconds on map 3 and visited 19 states.
13 moves were used: ULDDUUUURDDDD
Weighted A* average time:  0


In [1173]:
solve(4, "BFS")
print("BFS average time: ",0) # 0 ,0 ,0
solve(4, "DFS")
print("DFS average time: ", 0) # 0 ,0 ,0
solve(4, "IDS")
print("IDS average time: ", 0) # 0 ,0 ,0
solve(4, "A*")
print("A* average time: ", 0) # 0 ,0 ,0
solve(4, "Weighted A*")
print("Weighted A* average time: ", 0) # 0 ,0 ,0

BFS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
BFS average time:  0
DFS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
DFS average time:  0
IDS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
IDS average time:  0
A* took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
A* average time:  0
Weighted A* took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
Weighted A* average time:  0


In [1174]:
solve(5, "BFS")
print("BFS average time: ", 0.07) # 0.08 ,0.09 ,0.06
#solve(5, "DFS")
print("DFS average time: ", "-") # -
solve(5, "IDS")
print("IDS average time: ", 5.18) # 4.98 ,5.4 ,5.15
solve(5, "A*")
print("A* average time: ", 0.02) # 0.02 ,0.03 ,0.02
solve(5, "Weighted A*")
print("Weighted A* average time: ", 0) # 0 ,0 ,0

BFS took 0.06 seconds on map 5 and visited 5709 states.
15 moves were used: ULDDRDLLLUUURUL
BFS average time:  0.07
DFS average time:  -
IDS took 5.14 seconds on map 5 and visited 77488 states.
15 moves were used: ULDDRDLLLUUURUL
IDS average time:  5.18
A* took 0.03 seconds on map 5 and visited 838 states.
15 moves were used: LULDDRDLLUUURUL
A* average time:  0.02
Weighted A* took 0.0 seconds on map 5 and visited 45 states.
19 moves were used: LULDLRDLLUURRRDDDLL
Weighted A* average time:  0


In [1175]:
solve(6, "BFS")
print("BFS average time: ",0.37) # 0.41 ,0.32 ,0.37
#solve(6, "DFS")
print("DFS average time: ", '-') # -
#solve(6, "IDS")
print("IDS average time: ", '-') # -
solve(6, "A*")
print("A* average time: ", 0.3) # 0.28 ,0.27 ,0.33
solve(6, "Weighted A*")
print("Weighted A* average time: ", 0.01) # 0.01 ,0.01 ,0.1

BFS took 0.39 seconds on map 6 and visited 17531 states.
34 moves were used: UUUUURRRLLLLLLLDDDDDDDDDRDLRRRRRRR
BFS average time:  0.37
DFS average time:  -
IDS average time:  -
A* took 0.35 seconds on map 6 and visited 7036 states.
34 moves were used: RRDDDDDRLLLLLLLUUUUUUUUURULRRRRRRR
A* average time:  0.3
Weighted A* took 0.01 seconds on map 6 and visited 219 states.
34 moves were used: UUUUURRRLLLLLLLRDDDDDDDDDDLRRRRRRR
Weighted A* average time:  0.01


In [1176]:
solve(7, "BFS")
print("BFS average time: ",1.36) # 1.27 ,1.36 ,1.45
#solve(6, "DFS")
print("DFS average time: ", '-') # -
#solve(6, "IDS")
print("IDS average time: ", '-') # -
solve(7, "A*")
print("A* average time: ", 0.38) # 0.3 ,0.45 ,0.32
solve(7, "Weighted A*")
print("Weighted A* average time: ", 0.02) # 0.02 ,0.02 ,0.1

BFS took 1.44 seconds on map 7 and visited 50392 states.
34 moves were used: RURRDDDDLDRUUUULLLRDRDRDDLLDLLUUDR
BFS average time:  1.36
DFS average time:  -
IDS average time:  -
A* took 0.74 seconds on map 7 and visited 10475 states.
34 moves were used: RURRDDDDLDRUUUULLLRDRDRDDLLDLLURLU
A* average time:  0.38
Weighted A* took 0.02 seconds on map 7 and visited 655 states.
38 moves were used: RURRDLLLRRRDDDLDRUUUULLDRDRDDLLDLLUUDR
Weighted A* average time:  0.02


In [1179]:
solve(8, "BFS")
print("BFS average time: ", 0.07) # 0.08 ,0.08 ,0.05
#solve(6, "DFS")
print("DFS average time: ", '-') # -
#solve(6, "IDS")
print("IDS average time: ", '-') # -
solve(8, "A*")
print("A* average time: ", 0.14) # 0.27 ,0.06 ,0.5
solve(8, "Weighted A*")
print("Weighted A* average time: ", 0.05) # 0.1 ,0.03 ,0.02

BFS took 0.1 seconds on map 8 and visited 7333 states.
14 moves were used: UURDLDRRDRURDR
BFS average time:  0.07
DFS average time:  -
IDS average time:  -
A* took 0.08 seconds on map 8 and visited 1106 states.
14 moves were used: UURDLDRRDRURDR
A* average time:  0.14
Weighted A* took 0.04 seconds on map 8 and visited 736 states.
27 moves were used: URDRRRLLLUURRDLULDLDRRDRURD
Weighted A* average time:  0.05


In [1183]:
#solve(8, "BFS")
print("BFS average time: ", '-') # -
#solve(6, "DFS")
print("DFS average time: ", '-') # -
#solve(6, "IDS")
print("IDS average time: ", '-') # -
#solve(8, "A*")
print("A* average time: ", '-') # -
solve(9, "Weighted A*")
print("Weighted A* average time: ", 0.19) # 0.18 ,0.21 ,0.18

BFS average time:  -
DFS average time:  -
IDS average time:  -
A* average time:  -
Weighted A* took 0.21 seconds on map 9 and visited 5467 states.
81 moves were used: RRLURRUUUUUULLDLDRRURDDDDRRDDRDLDLURULDRRULDLLUUURRDLDRLLDRRUURURDRDLDLDLUULURURD
Weighted A* average time:  0.19


In [1187]:
solve(10, "BFS")
print("BFS average time: ", 0.07) # 3.9 ,3.91 ,3.51
#solve(6, "DFS")
print("DFS average time: ", '-') # -
#solve(6, "IDS")
print("IDS average time: ", '-') # -
solve(10, "A*")
print("A* average time: ", 4) # 3.89 ,4.31 ,3.79
solve(10, "Weighted A*")
print("Weighted A* average time: ", 0.04) # 0.04 ,0.05 ,0.04

BFS took 3.39 seconds on map 10 and visited 163050 states.
46 moves were used: RRRRRDRULURULLLULDRUUULDRDLDRRDRULURURDDRDLLLL
BFS average time:  0.07
DFS average time:  -
IDS average time:  -
A* took 3.91 seconds on map 10 and visited 60324 states.
46 moves were used: RRRRRDRULURULLLULLDRUULDRDLDRRDRULURURDDRDLLLL
A* average time:  4
Weighted A* took 0.04 seconds on map 10 and visited 944 states.
46 moves were used: RRRRRDRULURULLLULLDRUULDRDLDRRDRULURURDDRDLLLL
Weighted A* average time:  0.04
